In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


In [9]:
df = pd.read_csv("hospital_patients_clean.csv")
df.shape


(4947, 8)

In [7]:
df.columns


Index(['PatientID', 'Age', 'Gender', 'Diagnosis', 'AdmissionDate',
       'DischargeDate', 'HospitalID', 'LengthOfStay'],
      dtype='object')

In [10]:
X = df[["Age", "Gender", "Diagnosis", "HospitalID"]]
y = df["LengthOfStay"]


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Gender", "Diagnosis", "HospitalID"]
numeric_features = ["Age"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


In [14]:
import numpy as np
from sklearn.metrics import mean_squared_error

baseline_pred = np.full(len(y_test), y_train.mean())
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_rmse


np.float64(2.872498044107232)

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

linreg = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LinearRegression())
])

linreg.fit(X_train, y_train)
y_pred_lr = linreg.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

rmse_lr, r2_lr


(np.float64(2.900910016740617), -0.021439044568219767)

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

rmse_rf, r2_rf


(np.float64(3.068134829059165), -0.14259632001644684)

In [19]:
df["LongStay"] = (df["LengthOfStay"] >= 6).astype(int)
df["LongStay"].value_counts()


LongStay
0    2553
1    2394
Name: count, dtype: int64

In [20]:
X = df[["Age", "Gender", "Diagnosis", "HospitalID"]]
y = df["LongStay"]


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["Gender", "Diagnosis", "HospitalID"]
numeric_features = ["Age"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


In [23]:
import numpy as np
from sklearn.metrics import accuracy_score

most_common = y_train.mode()[0]
baseline_pred = np.full(len(y_test), most_common)

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_acc


0.5161616161616162

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

logreg = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logreg.fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
acc_lr


0.48383838383838385

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
acc_rf


0.4858585858585859

In [26]:
from sklearn.metrics import classification_report

print("Logistic Regression")
print(classification_report(y_test, y_pred_lr))

print("Random Forest")
print(classification_report(y_test, y_pred_rf))


Logistic Regression
              precision    recall  f1-score   support

           0       0.50      0.60      0.54       511
           1       0.46      0.36      0.41       479

    accuracy                           0.48       990
   macro avg       0.48      0.48      0.47       990
weighted avg       0.48      0.48      0.48       990

Random Forest
              precision    recall  f1-score   support

           0       0.50      0.54      0.52       511
           1       0.47      0.43      0.45       479

    accuracy                           0.49       990
   macro avg       0.48      0.48      0.48       990
weighted avg       0.48      0.49      0.48       990



In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_km = df[["Age", "LengthOfStay", "Diagnosis"]]

preprocess_km = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "LengthOfStay"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["Diagnosis"])
    ]
)


In [33]:
X_km_prep = preprocess_km.fit_transform(X_km)
X_km_prep.shape

(4947, 17)

In [34]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_km_prep)

df["Cluster"] = clusters
df["Cluster"].value_counts()


Cluster
2    1690
1    1634
0    1623
Name: count, dtype: int64

In [35]:
df.groupby("Cluster")[["Age", "LengthOfStay"]].agg(["mean", "median", "count"])


Age              LengthOfStay             
              mean median count         mean median count
Cluster                                                  
0        31.535428   33.0  1623     2.899569    3.0  1623
1        32.156059   34.0  1634     8.171971    8.0  1634
2        77.222485   78.0  1690     5.268639    5.0  1690

In [36]:
(df.groupby("Cluster")["Diagnosis"]
 .value_counts(normalize=True)
 .groupby(level=0)
 .head(5))


Cluster  Diagnosis              
0        Acute Bronchitis           0.081331
         Atrial Fibrillation        0.075786
         Cholelithiasis             0.075169
         Urinary Tract Infection    0.073321
         Osteoarthritis             0.070856
1        Atrial Fibrillation        0.080171
         Asthma                     0.075275
         Urinary Tract Infection    0.074051
         Myocardial Infarction      0.070991
         Diverticulitis             0.069767
2        Cholelithiasis             0.076331
         Osteoarthritis             0.076331
         Urinary Tract Infection    0.072189
         Chronic Kidney Disease     0.069822
         Diverticulitis             0.069822
Name: proportion, dtype: float64

In [37]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_rf)
cm


array([[274, 237],
       [272, 207]])

In [38]:
import pandas as pd

cm_df = pd.DataFrame(
    cm,
    index=["Actual_Normal", "Actual_LongStay"],
    columns=["Pred_Normal", "Pred_LongStay"]
)

cm_df


,Pred_Normal,Pred_LongStay
Actual_Normal,274,237
Actual_LongStay,272,207


In [39]:
cm_df / cm_df.values.sum()


,Pred_Normal,Pred_LongStay
Actual_Normal,0.276768,0.239394
Actual_LongStay,0.274747,0.209091


In [40]:
df.to_csv("hospital_patients_tableau.csv", index=False)
